# LLM-Generated Dataset Consistency Analysis

**How consistent are LLM-generated values and distributions across repeated runs under zero, one, and few-shot prompting?**

This notebook computes consistency metrics across repeated generations (run1/run2/run3) of LLM-generated datasets under different prompt conditions (zero-shot, one-shot, few-shot) for each domain. It reports:

- Numeric consistency: mean difference, std difference, KS statistic.
- Categorical consistency: Jaccard (unique overlap), entropy difference, frequency correlation.
- Semantic consistency: cosine similarity between embedding centroids of sampled texts.
- Overall consistency: average of the above sections.

## Imports

In [1]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

from data_loading import *
from consistency_metrics import *

# WARNINGS
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="pkg_resources")

In [ ]:
PROJECT_ROOT = find_project_root()
GENERATED_DIR = PROJECT_ROOT / "data" / "generated"
ANALYSIS_OUTPUT_DIR = PROJECT_ROOT / "analysis" / "consistency"
ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GROUND_TRUTH_PATH = PROJECT_ROOT / "data" / "preprocessed" / "column_type_ground_truth.json"
if GROUND_TRUTH_PATH.exists():
    with open(GROUND_TRUTH_PATH, 'r') as f:
        GROUND_TRUTH = json.load(f)
    print(f"\nLoaded ground truth for domains: {list(GROUND_TRUTH.keys())}")
else:
    GROUND_TRUTH = {}
    print(f"\n - No ground truth found at {GROUND_TRUTH_PATH}.")

RANDOM_STATE = 42
SEMANTIC_SAMPLE_SIZE = None 
EMBEDDING_METHOD = "sentence-transformers" 
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # using MiniLM for semantic stability

semantic_sample_desc = "all rows" if SEMANTIC_SAMPLE_SIZE is None else SEMANTIC_SAMPLE_SIZE

print(f"  Method: {EMBEDDING_METHOD}")
print(f"  Model: {EMBEDDING_MODEL_NAME}")
print(f"  Sample size per run: {semantic_sample_desc}")


Loaded ground truth for domains: ['hatecrime', 'employment', 'lending']
  Method: sentence-transformers
  Model: sentence-transformers/all-MiniLM-L6-v2
  Sample size per run: all rows


In [3]:
# discover all (model base × domain × shot) groups aggregated over multiple runs
group_map = load_group_dataframes(GENERATED_DIR)
print(f"Discovered {len(group_map)} (model, domain, shot) groups.")

list(group_map.keys())

Discovered 27 (model, domain, shot) groups.


[('kimi-k2-instruct-0905', 'employment', 'few'),
 ('kimi-k2-instruct-0905', 'employment', 'one'),
 ('kimi-k2-instruct-0905', 'employment', 'zero'),
 ('kimi-k2-instruct-0905', 'hatecrime', 'few'),
 ('kimi-k2-instruct-0905', 'hatecrime', 'one'),
 ('kimi-k2-instruct-0905', 'hatecrime', 'zero'),
 ('kimi-k2-instruct-0905', 'lending', 'few'),
 ('kimi-k2-instruct-0905', 'lending', 'one'),
 ('kimi-k2-instruct-0905', 'lending', 'zero'),
 ('llama-3.1-8b-instruct', 'employment', 'few'),
 ('llama-3.1-8b-instruct', 'employment', 'one'),
 ('llama-3.1-8b-instruct', 'employment', 'zero'),
 ('llama-3.1-8b-instruct', 'hatecrime', 'few'),
 ('llama-3.1-8b-instruct', 'hatecrime', 'one'),
 ('llama-3.1-8b-instruct', 'hatecrime', 'zero'),
 ('llama-3.1-8b-instruct', 'lending', 'few'),
 ('llama-3.1-8b-instruct', 'lending', 'one'),
 ('llama-3.1-8b-instruct', 'lending', 'zero'),
 ('qwen3-coder-30b-a3b-instruct', 'employment', 'few'),
 ('qwen3-coder-30b-a3b-instruct', 'employment', 'one'),
 ('qwen3-coder-30b-a3b-i

In [4]:
# computing stability metrics per group
summary_rows = []
per_group_details = {}

for (model_base, domain, shot), dfs in group_map.items():
    run_row_counts = [len(df) for df in dfs]
    num_runs = len(dfs)
    
    numeric_df, categorical_df, semantic_df, index_dict = compute_all_stability(
        dfs,
        semantic_sample_size=SEMANTIC_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        embedding_method=EMBEDDING_METHOD,
        embedding_model_name=EMBEDDING_MODEL_NAME,
        domain=domain, 
        ground_truth=GROUND_TRUTH,
    )

    per_group_details[(model_base, domain, shot)] = {
        "numeric": numeric_df,
        "categorical": categorical_df,
        "semantic": semantic_df,
        "index": index_dict,
    }

    row_info = {}
    for i in range(num_runs):
        row_info[f"num_rows_run{i+1}"] = run_row_counts[i]
    for i in range(num_runs, 3):
        row_info[f"num_rows_run{i+1}"] = 0
    
    summary_rows.append(
        {
            "model_base": model_base,
            "domain": domain,
            "shot": shot,
            **row_info,
            "avg_rows_per_run": np.mean(run_row_counts).round(1) if run_row_counts else 0,
            "numeric_score": index_dict.get("numeric"),
            "categorical_score": index_dict.get("categorical"),
            "semantic_score": index_dict.get("semantic"),
            "overall_stability": index_dict.get("overall"),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(by=["domain", "shot", "overall_stability"], ascending=[True, True, False])
summary_df.reset_index(drop=True, inplace=True)

summary_output_path = ANALYSIS_OUTPUT_DIR / "per_group_consistency.csv"
summary_df.to_csv(summary_output_path, index=False)

summary_df.sort_values(by="overall_stability", ascending=False)

/Users/veronhoxha/Desktop/master_thesis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,model_base,domain,shot,num_rows_run1,num_rows_run2,num_rows_run3,avg_rows_per_run,numeric_score,categorical_score,semantic_score,overall_stability
9,kimi-k2-instruct-0905,hatecrime,few,1004,1007,1012,1007.7,0.945154,0.926449,0.999962,0.957231
15,kimi-k2-instruct-0905,hatecrime,zero,1042,1026,1039,1035.7,0.843431,0.874932,0.999966,0.906203
16,qwen3-coder-30b-a3b-instruct,hatecrime,zero,1012,987,988,995.7,0.579845,0.804727,0.999960,0.795049
3,qwen3-coder-30b-a3b-instruct,employment,one,1020,1051,1019,1030.0,0.783562,0.849323,0.664052,0.765544
12,qwen3-coder-30b-a3b-instruct,hatecrime,one,1009,1019,1015,1014.3,0.440050,0.803949,0.999888,0.748215
0,kimi-k2-instruct-0905,employment,few,1031,1003,1044,1026.0,0.739693,0.783687,0.665272,0.729486
10,qwen3-coder-30b-a3b-instruct,hatecrime,few,1032,1031,1018,1027.0,0.335823,0.847492,0.999923,0.728018
6,qwen3-coder-30b-a3b-instruct,employment,zero,1010,1018,1027,1018.3,0.781186,0.768205,0.332496,0.627001
13,kimi-k2-instruct-0905,hatecrime,one,1000,998,1047,1015.0,0.000000,0.824416,0.999823,0.608472
11,llama-3.1-8b-instruct,hatecrime,few,441,237,334,337.3,0.000000,0.755221,0.998240,0.584901


In [5]:
# showing best and worst overall stability per domain and shot
if not summary_df.empty:
    display(summary_df.groupby(["domain", "shot"]).head(1).sort_values(by="overall_stability", ascending=False)) 
    print("\nLowest stability per (domain, shot):")
    display(summary_df.groupby(["domain", "shot"]).tail(1))
else:
    print("No groups found.")

,model_base,domain,shot,num_rows_run1,num_rows_run2,num_rows_run3,avg_rows_per_run,numeric_score,categorical_score,semantic_score,overall_stability
9,kimi-k2-instruct-0905,hatecrime,few,1004,1007,1012,1007.7,0.945154,0.926449,0.999962,0.957231
15,kimi-k2-instruct-0905,hatecrime,zero,1042,1026,1039,1035.7,0.843431,0.874932,0.999966,0.906203
3,qwen3-coder-30b-a3b-instruct,employment,one,1020,1051,1019,1030.0,0.783562,0.849323,0.664052,0.765544
12,qwen3-coder-30b-a3b-instruct,hatecrime,one,1009,1019,1015,1014.3,0.440050,0.803949,0.999888,0.748215
0,kimi-k2-instruct-0905,employment,few,1031,1003,1044,1026.0,0.739693,0.783687,0.665272,0.729486
6,qwen3-coder-30b-a3b-instruct,employment,zero,1010,1018,1027,1018.3,0.781186,0.768205,0.332496,0.627001
21,kimi-k2-instruct-0905,lending,one,1011,1030,1014,1018.3,0.655198,0.795991,0.000000,0.483246
18,qwen3-coder-30b-a3b-instruct,lending,few,456,630,629,571.7,0.521496,0.611706,0.000000,0.377356
24,qwen3-coder-30b-a3b-instruct,lending,zero,513,703,764,660.0,0.173234,0.597703,0.000000,0.256722



Lowest stability per (domain, shot):


,model_base,domain,shot,num_rows_run1,num_rows_run2,num_rows_run3,avg_rows_per_run,numeric_score,categorical_score,semantic_score,overall_stability
2,llama-3.1-8b-instruct,employment,few,955,961,946,954.0,0.000000,0.650467,0.664731,0.438626
5,llama-3.1-8b-instruct,employment,one,939,1011,933,961.0,0.000000,0.671216,0.663492,0.445121
8,llama-3.1-8b-instruct,employment,zero,986,791,952,909.7,0.000000,0.562280,0.330423,0.297601
11,llama-3.1-8b-instruct,hatecrime,few,441,237,334,337.3,0.000000,0.755221,0.998240,0.584901
14,llama-3.1-8b-instruct,hatecrime,one,569,733,351,551.0,0.000000,0.694607,0.998009,0.564639
17,llama-3.1-8b-instruct,hatecrime,zero,830,749,777,785.3,0.000000,0.701215,0.994880,0.565794
20,kimi-k2-instruct-0905,lending,few,834,704,581,706.3,0.000000,0.707557,0.000000,0.235616
23,llama-3.1-8b-instruct,lending,one,967,1013,975,985.0,0.000000,0.748788,0.000000,0.249346
26,kimi-k2-instruct-0905,lending,zero,1040,863,830,911.0,0.320087,0.258209,0.000000,0.192572


In [6]:
print("\n" + "="*100)
print("COMPREHENSIVE COLUMN TYPE SUMMARY")
print("="*100)

for (model_base, domain, shot), dfs in group_map.items():
    # detect column types
    numeric_cols, categorical_cols, text_cols, _ = detect_column_types(dfs)
    
    all_cols_per_run = [set(df.columns) for df in dfs]
    common_cols = set.intersection(*all_cols_per_run)
    all_cols = set.union(*all_cols_per_run)
    unique_cols_per_run = [len(cols) for cols in all_cols_per_run]
    
    # get summary score
    summary_row = summary_df[(summary_df['model_base'] == model_base) & 
                             (summary_df['domain'] == domain) & 
                             (summary_df['shot'] == shot)]
    
    print(f"\n{'='*100}")
    print(f"Model: {model_base} | Domain: {domain} | Shot: {shot}")
    print(f"{'='*100}")
    
    # schema consistency check
    if len(set(unique_cols_per_run)) > 1 or len(common_cols) < len(all_cols):
        print(f"  WARNING: Column counts vary across runs: {unique_cols_per_run}")
        print(f"   - Only {len(common_cols)} common columns used for consistency analysis")
        print(f"   - Missing columns in some runs: {len(all_cols) - len(common_cols)}")
    else:
        print(f"Schema consistent across runs: {unique_cols_per_run[0]} columns")
    
    # column type breakdown
    print(f"\n- DETECTED COLUMN TYPES:")
    print(f"   - Numeric:     {len(numeric_cols):2d} columns")
    print(f"   - Categorical: {len(categorical_cols):2d} columns")
    print(f"   - Text:        {len(text_cols):2d} columns")
    print(f"   - Total:       {len(numeric_cols) + len(categorical_cols) + len(text_cols)} columns")
    
    # show actual column names
    if numeric_cols:
        print(f"\n   Numeric columns: {numeric_cols}")
    if categorical_cols:
        print(f"\n   Categorical columns: {categorical_cols}")
    if text_cols:
        print(f"\n   Text columns: {text_cols}")
    
    if not summary_row.empty:
        num_score = summary_row['numeric_score'].values[0]
        cat_score = summary_row['categorical_score'].values[0]
        sem_score = summary_row['semantic_score'].values[0]
        overall = summary_row['overall_stability'].values[0]
        
        print(f"\n- CONSISTENCY SCORES:")
        print(f"   - Numeric:     {num_score:.4f} {'(No numeric columns detected)' if pd.isna(num_score) and len(numeric_cols) == 0 else ''}")
        print(f"   - Categorical: {cat_score:.4f} {'(No categorical columns detected)' if pd.isna(cat_score) and len(categorical_cols) == 0 else ''}")
        print(f"   - Semantic:    {sem_score:.4f} {'(No text columns detected)' if pd.isna(sem_score) and len(text_cols) == 0 else '(All text columns identical/empty across runs)' if pd.isna(sem_score) and len(text_cols) > 0 else ''}")
        print(f"   - Overall:     {overall:.4f}")
    
    print()


COMPREHENSIVE COLUMN TYPE SUMMARY

Model: kimi-k2-instruct-0905 | Domain: employment | Shot: few
Schema consistent across runs: 18 columns

- DETECTED COLUMN TYPES:
   - Numeric:     12 columns
   - Categorical:  4 columns
   - Text:         2 columns
   - Total:       18 columns

   Numeric columns: ['DiffMeanBonusPercent', 'DiffMeanHourlyPercent', 'DiffMedianBonusPercent', 'FemaleBonusPercent', 'FemaleLowerMiddleQuartile', 'FemaleLowerQuartile', 'FemaleUpperMiddleQuartile', 'MaleBonusPercent', 'MaleLowerMiddleQuartile', 'MaleLowerQuartile', 'MaleTopQuartile', 'MaleUpperMiddleQuartile']

   Categorical columns: ['DiffMedianHourlyPercent', 'EmployerSize', 'FemaleTopQuartile', 'SicCodes']

   Text columns: ['DateSubmitted', 'EmployerName']

- CONSISTENCY SCORES:
   - Numeric:     0.7397 
   - Categorical: 0.7837 
   - Semantic:    0.6653 
   - Overall:     0.7295


Model: kimi-k2-instruct-0905 | Domain: employment | Shot: one
Schema consistent across runs: 18 columns

- DETECTED COLUMN

In [7]:
print("\n" + "=" * 80)
print("Column Detection Examples")
print("=" * 80)

example_key = list(group_map.keys())[0]
example_dfs = group_map[example_key]
print(f"Using example group: {example_key}")

common_cols = set(example_dfs[0].columns)
for df in example_dfs[1:]:
    common_cols &= set(df.columns)

examples = {"numeric": None, "categorical": None, "text": None}

for col in sorted(common_cols):
    unique_counts = [df[col].astype(str).dropna().nunique() for df in example_dfs]
    lengths = [len(df[col].astype(str).dropna()) for df in example_dfs]
    max_unique = max(unique_counts)
    max_len = max(lengths)
    unique_fraction = (max_unique / max_len) if max_len > 0 else 0
    is_numeric = all(pd.api.types.is_numeric_dtype(df[col]) for df in example_dfs)

    if is_numeric and examples["numeric"] is None:
        examples["numeric"] = col
    elif (max_unique <= 50 or unique_fraction <= 0.5) and examples["categorical"] is None:
        examples["categorical"] = col
    elif (max_unique > 50 and unique_fraction > 0.5) and examples["text"] is None:
        examples["text"] = col

    if all(examples.values()):
        break

for col_type in ["numeric", "categorical", "text"]:
    example_col = examples.get(col_type)
    if not example_col:
        print(f"\nNo clear {col_type} example found in this group.")
        continue

    print(f"\n--- {col_type.upper()} column: {example_col} ---")
    for run_idx, df in enumerate(example_dfs, start=1):
        s = df[example_col].astype(str).dropna()
        unique_count = s.nunique()
        total_count = len(s)
        unique_fraction = (unique_count / total_count) if total_count > 0 else 0
        dtype = df[example_col].dtype
        print(
            f"  Run {run_idx}: {unique_count:4d} unique values out of {total_count:4d} rows "
            f"(fraction={unique_fraction:.2f}) | dtype={dtype}"
        )

    max_unique = max(df[example_col].astype(str).dropna().nunique() for df in example_dfs)
    max_len = max(len(df[example_col].astype(str).dropna()) for df in example_dfs)
    frac = (max_unique / max_len) if max_len > 0 else 0
    numeric_flag = all(pd.api.types.is_numeric_dtype(df[example_col]) for df in example_dfs)

    if numeric_flag:
        decision = "NUMERIC (pandas numeric dtype in every run)"
    elif max_unique <= 50 or frac <= 0.5:
        decision = f"CATEGORICAL (max_unique={max_unique} <= 50 or fraction={frac:.2f} <= 0.5)"
    else:
        decision = f"TEXT (max_unique={max_unique} > 50 and fraction={frac:.2f} > 0.5)"

    print(f"- Detection decision: {decision}")


Column Detection Examples
Using example group: ('kimi-k2-instruct-0905', 'employment', 'few')

--- NUMERIC column: DiffMeanBonusPercent ---
  Run 1:  370 unique values out of 1031 rows (fraction=0.36) | dtype=float64
  Run 2:  406 unique values out of 1003 rows (fraction=0.40) | dtype=float64
  Run 3:  384 unique values out of 1044 rows (fraction=0.37) | dtype=float64
- Detection decision: NUMERIC (pandas numeric dtype in every run)

--- CATEGORICAL column: DiffMeanHourlyPercent ---
  Run 1:  222 unique values out of 1031 rows (fraction=0.22) | dtype=float64
  Run 2:  239 unique values out of 1003 rows (fraction=0.24) | dtype=float64
  Run 3:  256 unique values out of 1044 rows (fraction=0.25) | dtype=float64
- Detection decision: NUMERIC (pandas numeric dtype in every run)

--- TEXT column: DateSubmitted ---
  Run 1: 1016 unique values out of 1031 rows (fraction=0.99) | dtype=object
  Run 2:  986 unique values out of 1003 rows (fraction=0.98) | dtype=object
  Run 3: 1017 unique value

In [8]:
if not summary_df.empty:
    print("\n1. OVERALL CONSISTENCY BY SHOT TYPE (All Models & Domains)")
    print("-" * 80)
    
    shot_summary = summary_df.groupby('shot').agg({
        'overall_stability': ['mean', 'std', 'min', 'max', 'count'],
        'numeric_score': 'mean',
        'categorical_score': 'mean',
        'semantic_score': 'mean',
    }).round(4)
    
    shot_summary.columns = ['_'.join(col).strip() if col[1] else col[0] for col in shot_summary.columns.values]
    shot_summary = shot_summary.rename(columns={
        'overall_stability_mean': 'Mean Overall',
        'overall_stability_std': 'Std Dev',
        'overall_stability_min': 'Min',
        'overall_stability_max': 'Max',
        'overall_stability_count': 'N Groups',
        'numeric_score_mean': 'Mean Numeric',
        'categorical_score_mean': 'Mean Categorical',
        'semantic_score_mean': 'Mean Semantic',
    })
    
    shot_order = ['zero', 'one', 'few']
    shot_summary = shot_summary.reindex([s for s in shot_order if s in shot_summary.index])
    
    display(shot_summary)

    shot_summary_path = ANALYSIS_OUTPUT_DIR / "shot_type_consistency.csv"
    shot_summary.reset_index().to_csv(shot_summary_path, index=False)
    
    # summary by domain and shot type
    print("\n2. CONSISTENCY BY DOMAIN AND SHOT TYPE")
    print("-" * 80)
    
    domain_shot_summary = summary_df.groupby(['domain', 'shot']).agg({
        'overall_stability': ['mean', 'std', 'count'],
        'numeric_score': 'mean',
        'categorical_score': 'mean',
        'semantic_score': 'mean',
    }).round(4)
    
    domain_shot_summary.columns = ['_'.join(col).strip() if col[1] else col[0] for col in domain_shot_summary.columns.values]
    domain_shot_summary = domain_shot_summary.rename(columns={
        'overall_stability_mean': 'Mean Overall',
        'overall_stability_std': 'Std Dev',
        'overall_stability_count': 'N Groups',
        'numeric_score_mean': 'Mean Numeric',
        'categorical_score_mean': 'Mean Categorical',
        'semantic_score_mean': 'Mean Semantic',
    })
    
    display(domain_shot_summary)

    domain_shot_summary_path = ANALYSIS_OUTPUT_DIR / "domain_shot_consistency.csv"
    domain_shot_summary.reset_index().to_csv(domain_shot_summary_path, index=False)
    
    print("\n3. CONSISTENCY BY MODEL (LLM overall)")
    print("-" * 80)
    
    model_summary = summary_df.groupby('model_base').agg({
        'overall_stability': ['mean', 'std', 'min', 'max', 'count'],
        'numeric_score': 'mean',
        'categorical_score': 'mean',
        'semantic_score': 'mean',
    }).round(4)
    
    model_summary.columns = ['_'.join(col).strip() if col[1] else col[0] for col in model_summary.columns.values]
    model_summary = model_summary.rename(columns={
        'overall_stability_mean': 'Mean Overall',
        'overall_stability_std': 'Std Dev',
        'overall_stability_min': 'Min',
        'overall_stability_max': 'Max',
        'overall_stability_count': 'N Groups',
        'numeric_score_mean': 'Mean Numeric',
        'categorical_score_mean': 'Mean Categorical',
        'semantic_score_mean': 'Mean Semantic',
    })
    
    display(model_summary)
    
    model_summary_path = ANALYSIS_OUTPUT_DIR / "model_consistency.csv"
    model_summary.reset_index().to_csv(model_summary_path, index=False)
    
    
    # model-specific patterns
    if summary_df['model_base'].nunique() > 1:
        print("\n5. CONSISTENCY BY MODEL AND SHOT TYPE")
        print("-" * 80)
        
        model_shot_summary = summary_df.groupby(['model_base', 'shot']).agg({
            'overall_stability': 'mean',
        }).round(4).unstack(level=1)
        
        model_shot_summary.columns = model_shot_summary.columns.droplevel(0)
        model_shot_summary = model_shot_summary[[s for s in shot_order if s in model_shot_summary.columns]]
        model_shot_summary.columns.name = None
        model_shot_summary = model_shot_summary.rename_axis('Model', axis=0)
        
        display(model_shot_summary)
 
        model_shot_summary_path = ANALYSIS_OUTPUT_DIR / "model_shot_consistency.csv"
        model_shot_summary.to_csv(model_shot_summary_path, index=True)
 
        print("\n6. CONSISTENCY BY MODEL AND DOMAIN")
        print("-" * 80)
        domain_order = ["hatecrime", "employment", "lending"]
 
        model_domain_summary = summary_df.groupby(['model_base', 'domain']).agg({
            'overall_stability': 'mean',
        }).round(4).unstack(level=1)
 
        model_domain_summary.columns = model_domain_summary.columns.droplevel(0)
        model_domain_summary = model_domain_summary[[d for d in domain_order if d in model_domain_summary.columns]]
        model_domain_summary.columns.name = None
        model_domain_summary = model_domain_summary.rename_axis('Model', axis=0)
 
        display(model_domain_summary)
 
        model_domain_summary_path = ANALYSIS_OUTPUT_DIR / "model_domain_consistency.csv"
        model_domain_summary.to_csv(model_domain_summary_path, index=True)
 
        print("\n7. CONSISTENCY BY MODEL × DOMAIN × SHOT")
        print("-" * 80)
        model_domain_shot_summary = (
            summary_df.pivot_table(
                index="model_base",
                columns=["domain", "shot"],
                values="overall_stability",
                aggfunc="mean",
            )
            .round(4)
        )
 
        column_order = []
        for dom in domain_order:
            for shot in shot_order:
                col_key = (dom, shot)
                if col_key in model_domain_shot_summary.columns:
                    column_order.append(col_key)
        if column_order:
            model_domain_shot_summary = model_domain_shot_summary[column_order]
 
        model_domain_shot_summary.index.name = "Model"
        model_domain_shot_summary.columns.names = ["Domain", "Shot"]
 
        display(model_domain_shot_summary)
 
        model_domain_shot_path = ANALYSIS_OUTPUT_DIR / "model_domain_shot_consistency.csv"
        model_domain_shot_summary.to_csv(model_domain_shot_path)
 
else:
    print("No summary data available.")


1. OVERALL CONSISTENCY BY SHOT TYPE (All Models & Domains)
--------------------------------------------------------------------------------


,Mean Overall,Std Dev,Min,Max,N Groups,Mean Numeric,Mean Categorical,Mean Semantic
shot,,,,,,,,
zero,0.4801,0.2594,0.1926,0.9062,9,0.2998,0.6608,0.4797
one,0.5221,0.1714,0.2493,0.7655,9,0.2489,0.7628,0.5545
few,0.5314,0.2403,0.2356,0.9572,9,0.2825,0.7568,0.5549



2. CONSISTENCY BY DOMAIN AND SHOT TYPE
--------------------------------------------------------------------------------


Mean Overall  Std Dev  N Groups  Mean Numeric  \
domain     shot                                                  
employment few         0.5485   0.1579         3        0.2466   
           one         0.5679   0.1729         3        0.2612   
           zero        0.4577   0.1649         3        0.2604   
hatecrime  few         0.7567   0.1878         3        0.4270   
           one         0.6404   0.0959         3        0.1467   
           zero        0.7557   0.1736         3        0.4744   
lending    few         0.2889   0.0771         3        0.1738   
           one         0.3580   0.1178         3        0.3387   
           zero        0.2269   0.0323         3        0.1644   

                 Mean Categorical  Mean Semantic  
domain     shot                                   
employment few             0.7334         0.6652  
           one             0.7778         0.6643  
           zero            0.6717         0.4410  
hatecrime  few             0.8431         0.9994  
           one             0.7743         0.9992  
           zero            0.7936         0.9983  
lending    few             0.6939         0.0000  
           one             0.7364         0.0000  
           zero            0.5171         0.0000


3. CONSISTENCY BY MODEL (LLM overall)
--------------------------------------------------------------------------------


...

,zero,one,few
Model,,,
kimi-k2-instruct-0905,0.5157,0.5282,0.6408
llama-3.1-8b-instruct,0.3650,0.4197,0.4258
qwen3-coder-30b-a3b-instruct,0.5596,0.6184,0.5276



6. CONSISTENCY BY MODEL AND DOMAIN
--------------------------------------------------------------------------------


,hatecrime,employment,lending
Model,,,
kimi-k2-instruct-0905,0.8240,0.5569,0.3038
llama-3.1-8b-instruct,0.5718,0.3938,0.2449
qwen3-coder-30b-a3b-instruct,0.7571,0.6233,0.3251



7. CONSISTENCY BY MODEL × DOMAIN × SHOT
--------------------------------------------------------------------------------


Domain                       hatecrime                 employment          \
Shot                              zero     one     few       zero     one   
Model                                                                       
kimi-k2-instruct-0905           0.9062  0.6085  0.9572     0.4484  0.4929   
llama-3.1-8b-instruct           0.5658  0.5646  0.5849     0.2976  0.4451   
qwen3-coder-30b-a3b-instruct    0.7950  0.7482  0.7280     0.6270  0.7655   

Domain                               lending                  
Shot                             few    zero     one     few  
Model                                                         
kimi-k2-instruct-0905         0.7295  0.1926  0.4832  0.2356  
llama-3.1-8b-instruct         0.4386  0.2315  0.2493  0.2538  
qwen3-coder-30b-a3b-instruct  0.4774  0.2567  0.3414  0.3774

The consistency metrics measure how similar the generated datasets are across repeated runs.

A score of 1.0 indicates perfect consistency (identical distributions across runs).
A score close to 0.0 indicates high variability (very different distributions across runs).

• **Numeric consistency:** Measures distribution similarity for numeric columns
  - Based on: mean differences, std differences, Kolmogorov-Smirnov statistic
  
• **Categorical consistency:** Measures similarity of categorical value sets and distributions
  - Based on: Jaccard similarity, entropy differences, frequency correlations
  
• **Semantic consistency:** Measures semantic similarity of text content
  - Based on: cosine similarity of sentence embeddings (all-MiniLM-L6-v2)
  
• **Overall consistency:** Weighted average of the three metrics
  - Penalizes type mismatches

Higher overall consistency indicates that the LLM produces more consistent outputs across repeated generations when using that particular shot type.